In [1]:
# ============================================================
# SAM3 — Multi-Category Instance Segmentation
#         KITTI Eigen-Split Batch Runner
#
# DATASET LAYOUT (KITTI eigen split):
#   input_root/
#   └── <seq_date_drive>/          e.g. 2011_09_26_drive_0001_sync/
#       └── <date>/                e.g. 2011_09_26/
#           └── <seq>/             e.g. 0000000001/
#               └── image_02/
#                   └── data/
#                       ├── 0000000000.png
#                       ├── 0000000001.png
#                       └── ...
#
# LiDAR ground-truth depth (same relative key, different subtree):
#   input_root/*/*/*/proj_depth/groundtruth/image_02/*.png
#   (not used for segmentation but preserved for reference)
#
# OUTPUT (mirrors input structure exactly):
#   output_root/
#   └── <seq_date_drive>/<date>/<seq>/image_02/data/
#       └── <frame_stem>/          e.g. 0000000005/
#           ├── 0000000005_vehicle_vehicle_headlights_0000_<uid8>_0.923_mask.png
#           ├── 0000000005_pedestrian_pedestrian_0001_<uid8>_0.791_mask.png
#           ├── 0000000005_overview.jpg      (if save_vis=True)
#           └── 0000000005_manifest.json     (instance index for this frame)
#
# Segmentation targets:
#   • Vehicles   – ONLY those with visible headlights / taillights
#   • Pedestrians
#   • Stationary objects – trees, poles, buildings, ad-boards,
#                          traffic signs
#
# File naming convention:
#   <frame_stem>_<super_cat>_<sub_cat>_<local_idx:04d>_<uid8>_<score:.3f>_mask.png
#
#   local_idx  : resets to 0000 for each frame (per-image ordering)
#   uid8       : first 8 hex chars of a UUID4 — globally unique across
#                ALL frames, ALL runs; safe to use as a foreign key
#
# Manifest JSON (per frame):
#   [
#     {
#       "uid":       "a3f2c1b0",        # unique instance ID
#       "local_idx": 0,
#       "super_cat": "vehicle",
#       "sub_cat":   "vehicle_headlights",
#       "score":     0.923,
#       "box":       [x1, y1, x2, y2],
#       "mask_file": "0000000005_vehicle_vehicle_headlights_0000_a3f2c1b0_0.923_mask.png"
#     },
#     ...
#   ]
# ============================================================


In [2]:

import json
import logging
import os
import shutil
import sys
import uuid
from pathlib import Path
from tqdm.auto import tqdm

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image


In [3]:

# ── SAM3 imports ──────────────────────────────────────────────
# Prerequisites:
#   pip install -e "git+https://github.com/facebookresearch/sam3.git#egg=sam3"
#   hf auth login   (after requesting access at huggingface.co/facebook/sam3)
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor



/STORAGE/Arka/MDE_accuracy_eval/sam3/sam3/model_builder.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/STORAGE/Arka/conda_envs/sam3_da3/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [4]:

# ═════════════════════════════════════════════════════════════
# 1.  LOGGING
# ═════════════════════════════════════════════════════════════

logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt = "%H:%M:%S",
)
log = logging.getLogger("sam3_kitti_eigen")


In [5]:
# ═════════════════════════════════════════════════════════════
# 2.  CONFIGURATION
# ═════════════════════════════════════════════════════════════

DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
SCORE_THRESHOLD = 0.40          # Drop detections below this confidence
IOU_THRESHOLD   = 0.60          # Greedy-NMS overlap threshold
CACHE_DIR       = "/STORAGE/Arka/cache"

# Extensions treated as images when scanning
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".ppm"}

# Glob pattern for RGB images in the KITTI eigen split layout.
EIGEN_GLOB_PATTERN = "*/*/*/image_02/data/*.png"

# ── Text prompts per sub-category ─────────────────────────────
#
# Prompt engineering principles applied
# ─────────────────────────────────────
#  1. PART-AS-SUBJECT  : In vehicle_lights and vehicle_wheels, the
#     measured part (headlight cluster, wheel) must be the grammatical
#     subject so SAM3 segments the part, not the whole vehicle.
#
#  2. ILLUMINATION MIX : Each light sub-category includes both
#     "active light source" prompts (night/dusk) AND "housing/assembly"
#     prompts (daytime) so the detector works across all lighting
#     conditions found in KITTI.
#
#  3. STATIC GEOMETRY  : Motion verbs ("approaching", "driving away",
#     "receding") are replaced with static camera-relative descriptors
#     ("front view of", "rear view of", "in profile") because SAM3
#     evaluates one frame with no temporal context.
#
#  4. ABSOLUTE DIRECTIONS : "nearside" / "offside" are replaced with
#     "left-side" / "right-side" (camera-relative), which are
#     unambiguous in training data regardless of country driving side.
#
#  5. NOUN-FIRST ANIMALS : Action verbs ("pecking", "running") are
#     avoided in favour of noun+location anchors so the model segments
#     a stable object region rather than latching on to a pose.
#
# Super-category taxonomy
# ───────────────────────
#  vehicle              – full vehicle body (with lights for detection anchor)
#  vehicle_lights       – isolated light clusters  (width / length measurement)
#  vehicle_wheels       – isolated wheel clusters  (width / length measurement)
#  cyclist              – bicycle+rider and bicycle alone
#  pedestrian           – on-foot persons
#  road                 – drivable surface (road-width estimation)
#  animal               – all animals on / near the road
#  stationary_object    – everything static beside / above the road
#
CATEGORY_PROMPTS: dict[str, list[str]] = {

    # ══════════════════════════════════════════════════════════
    # A.  VEHICLE LIGHTS
    # ══════════════════════════════════════════════════════════

    # ── A1. Both headlights  →  front-facing vehicle WIDTH ────
    "vehicle_both_headlights": [
        "both headlights on the front of a vehicle",
    ],

    # ── A2. Both taillights  →  rear-facing vehicle WIDTH ─────
    "vehicle_both_taillights": [
        "both taillights on the rear of a vehicle",
    ],

    # ── A3. Single headlight  →  profile vehicle LENGTH ───────
    "vehicle_single_headlight": [
        "single headlight of a vehicle in side profile",
    ],

    # ── A4. Single taillight  →  profile vehicle LENGTH ───────
    "vehicle_single_taillight": [
        "single taillight of a vehicle in side profile",
    ],

    # ══════════════════════════════════════════════════════════
    # B.  EXPANDED VEHICLE TYPES
    #     Subject = full vehicle body; lights used as detection
    #     anchor.  Motion verbs replaced with static view framing.
    # ══════════════════════════════════════════════════════════

    "personal_vehicle_headlights": [
        "vehicle with headlights",
    ],
    "personal_vehicle_taillights": [
        "vehicle with taillights",
    ],

    # ══════════════════════════════════════════════════════════
    # C.  VEHICLE WHEELS
    #     PART-AS-SUBJECT throughout so SAM3 masks the wheel(s),
    #     not the vehicle body.
    #     "nearside" / "offside" replaced with "left-side" /
    #     "right-side" (camera-relative, unambiguous globally).
    # ══════════════════════════════════════════════════════════

    # ── C1. Both front wheels  →  front-facing vehicle WIDTH ──
    "vehicle_both_front_wheels": [
        "both front wheels of a vehicle",
    ],

    # ── C2. Both rear wheels  →  rear-facing vehicle WIDTH ────
    "vehicle_both_rear_wheels": [
        "both rear wheels of a vehicle",
    ],

    # ── C3. Left/right front wheel  →  profile vehicle LENGTH ─
    "vehicle_single_front_wheel": [
        "single front wheel of a vehicle",
    ],

    # ── C4. Left/right rear wheel  →  profile vehicle LENGTH ──
    "vehicle_single_rear_wheel": [
        "single rear wheel of a vehicle",
    ],

    # ── C5. Both side wheels  →  full lateral vehicle LENGTH ──
    "vehicle_sideways_both_wheels": [
        "front and rear wheels of a vehicle in full side profile",
    ],

    # ══════════════════════════════════════════════════════════
    # D.  CYCLIST
    # ══════════════════════════════════════════════════════════

    "bicycle_with_rider": [
        "bicycle with a rider on the road",
        "bicycle and rider",
    ],
    "bicycle_only": [
        "bicycle parked on the roadside",
    ],

    # ══════════════════════════════════════════════════════════
    # E.  PEDESTRIAN
    # ══════════════════════════════════════════════════════════

    "pedestrian": [
        "person on the street",
    ],

    # ══════════════════════════════════════════════════════════
    # F.  ROAD SURFACE
    # ══════════════════════════════════════════════════════════

    "road_surface": [
        "road surface",
        "drivable road",
        "paved road",
        "asphalt road",
        "road carriageway",
        "street surface",
        "road lane",
        "highway surface",
        "road with lane markings",
        "tarmac road",
        
    ],

    # ══════════════════════════════════════════════════════════
    # G.  STATIONARY OBJECTS
    # ══════════════════════════════════════════════════════════

    # ── G1. Vegetation ────────────────────────────────────────
    "tree": [
        "tree trunk",
        "tree",
    ],

    # ── G2. Poles and columns ─────────────────────────────────
    "electric_pole": [
        "street light pole",
        "electric pole",
    ],
    "street_lamp": [
        "street lamp",
        "street light",
        "lamp post",
    ],
    "traffic_light_pole": [
        "traffic light on a pole",
        "traffic signal mast",
    ],

    # ── G3. Barriers and dividers ─────────────────────────────
    "road_barrier": [
        "road barrier",
        "road divider block",
    ],
    "guardrail": [
        "guardrail beside the road",
    ],
    "fence": [
        "fence beside the road",
    ],
    "road_divider": [
        "road divider",
    ],

    # ── G4. Road furniture ────────────────────────────────────
    "traffic_sign": [
        "traffic sign",
    ],
    "road_marking": [
        "zebra crossing markings",
    ],
    "speed_bump": [
        "speed bump on the road",
    ],
    "manhole_cover": [
        "manhole cover on the road",
    ],
    "fire_hydrant": [
        "fire hydrant on the roadside",
    ],
    "bench": [
        "bench beside the road",
    ],
    "trash_can": [
        "waste bin on the street",
    ],
    "parking_meter": [
        "parking meter on the pavement",
    ],

    # ── G5. Structures and infrastructure ─────────────────────
    "building": [
        "house beside the road",
        "building",
    ],
    "wall": [
        "wall beside the road",
    ],
    "bus_stop": [
        "bus stop on the roadside",
    ],
    "overhead_bridge": [
        "footbridge above the road",
    ],
    "tunnel_entrance": [
        "tunnel entrance on the road",
    ],

    # ── G6. Advertisement and signage ─────────────────────────
    "advertisement_board": [
        "billboard",
    ],
    "shop_signboard": [
        "shop sign above a storefront",
    ],

    # ── G7. Roadside infrastructure ───────────────────────────
    "sidewalk": [
        "sidewalk beside the road",
    ],
    "traffic_cone": [
        "orange traffic cone on the road",
    ],

    # ══════════════════════════════════════════════════════════
    # H.  ANIMALS
    #     Noun-first prompts throughout (no action verbs as the
    #     primary semantic anchor) for stable object segmentation.
    # ══════════════════════════════════════════════════════════

    # ── H1. Large wildlife ────────────────────────────────────
    "wildlife_large_small": [
        "animal on the road",
    ],

    # ── H2. Draught and working animals ──────────────────────
    "animal_drawn_vehicle": [
        "bullock cart on the road",
        "horse cart on the road",
        "donkey cart on the road",
        "camel cart on the road",
        "horse carriage on the road",
        "ox cart on the road",
        "mule with a load on the road",
    ],
}

# ── Map every sub-category → super-category ───────────────────
SUPER_CATEGORY_MAP: dict[str, str] = {
    # Vehicle lights (measurement-grade)
    "vehicle_both_headlights":          "vehicle_lights",
    "vehicle_both_taillights":          "vehicle_lights",
    "vehicle_single_headlight":         "vehicle_lights",
    "vehicle_single_taillight":         "vehicle_lights",
    # Personal transport
    "personal_vehicle_headlights":      "vehicle",
    "personal_vehicle_taillights":      "vehicle",
    # Vehicle wheels (measurement-grade)
    "vehicle_both_front_wheels":        "vehicle_wheels",
    "vehicle_both_rear_wheels":         "vehicle_wheels",
    "vehicle_single_front_wheel":       "vehicle_wheels",
    "vehicle_single_rear_wheel":        "vehicle_wheels",
    "vehicle_sideways_both_wheels":     "vehicle_wheels",
    # Cyclist
    "bicycle_with_rider":               "cyclist",
    "bicycle_only":                     "cyclist",
    # Pedestrian
    "pedestrian":                       "pedestrian",
    # Road surface
    "road_surface":                     "road",
    # Stationary objects – vegetation
    "tree":                             "stationary_object",
    # Stationary objects – poles
    "electric_pole":                    "stationary_object",
    "street_lamp":                      "stationary_object",
    "traffic_light_pole":               "stationary_object",
    # Stationary objects – barriers
    "road_barrier":                     "stationary_object",
    "guardrail":                        "stationary_object",
    "fence":                            "stationary_object",
    "road_divider":                     "stationary_object",
    # Stationary objects – road furniture
    "traffic_sign":                     "stationary_object",
    "road_marking":                     "stationary_object",
    "speed_bump":                       "stationary_object",
    "manhole_cover":                    "stationary_object",
    "fire_hydrant":                     "stationary_object",
    "bench":                            "stationary_object",
    "trash_can":                        "stationary_object",
    "parking_meter":                    "stationary_object",
    # Stationary objects – structures
    "building":                         "stationary_object",
    "wall":                             "stationary_object",
    "bus_stop":                         "stationary_object",
    "overhead_bridge":                  "stationary_object",
    "tunnel_entrance":                  "stationary_object",
    # Stationary objects – signage
    "advertisement_board":              "stationary_object",
    "shop_signboard":                   "stationary_object",
    # Stationary objects – roadside infra
    "sidewalk":                         "stationary_object",
    "traffic_cone":                     "stationary_object",
    # Animals
    "wildlife_large_small":             "animal",
    "animal_drawn_vehicle":             "animal",
    }

# ── Colours for visualisation overlay (RGB) ───────────────────
VIS_COLORS: dict[str, tuple[int, int, int]] = {
    "vehicle":           (  0, 200, 255),   # cyan
    "vehicle_lights":    (255, 240,   0),   # yellow   – measurement lights
    "vehicle_wheels":    (255, 160,   0),   # orange   – measurement wheels
    "cyclist":           (180,  80, 255),   # purple
    "pedestrian":        (255, 100,  50),   # orange-red
    "road":              ( 80, 180,  80),   # green    – road surface
    "animal":            (255,  60, 180),   # pink/magenta
    "stationary_object": (150, 230, 150),   # light green
}



In [6]:

# ═════════════════════════════════════════════════════════════
# 3.  MODEL LOADING
# ═════════════════════════════════════════════════════════════

def load_sam3(cache_dir: str = CACHE_DIR) -> tuple:
    """
    Load SAM3 image model and processor (call once, reuse everywhere).

    HuggingFace credentials must be set up beforehand:
        hf auth login
    """
    log.info("Loading SAM3 image model …")
    os.environ.setdefault("HF_HOME", cache_dir)
    model     = build_sam3_image_model()
    processor = Sam3Processor(model)
    log.info("SAM3 loaded on %s.", DEVICE)
    return model, processor



In [7]:

# ═════════════════════════════════════════════════════════════
# 4.  INFERENCE HELPERS
# ═════════════════════════════════════════════════════════════

def _query_sam3(processor: Sam3Processor,
                state: dict,
                prompt: str,
                score_thresh: float) -> list[dict]:
    """Single text-prompt pass through SAM3; returns filtered instances."""
    output  = processor.set_text_prompt(state=state, prompt=prompt)
    masks   = output["masks"]   # (N, 1, H, W) or (N, H, W)
    boxes   = output["boxes"]   # (N, 4)  xyxy
    scores  = output["scores"]  # (N,)

    hits = []
    for i in range(len(scores)):
        score = float(scores[i])
        if score < score_thresh:
            continue
        m = masks[i]
        if m.dim() == 3:
            m = m.squeeze(0)
        hits.append({
            "mask":   m.cpu().bool().numpy(),
            "box":    boxes[i].cpu().numpy(),
            "score":  score,
            "prompt": prompt,
        })
    return hits


def _mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    """Intersection-over-Union for two boolean masks."""
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / (float(union) + 1e-6)


def _deduplicate(instances: list[dict],
                 iou_thresh: float) -> list[dict]:
    """
    Greedy score-sorted NMS within a super-category bucket.
    Removes overlapping duplicates produced by multiple prompts.
    """
    instances = sorted(instances, key=lambda x: x["score"], reverse=True)
    kept: list[dict] = []
    for cand in instances:
        if not any(_mask_iou(cand["mask"], k["mask"]) > iou_thresh
                   for k in kept):
            kept.append(cand)
    return kept



In [8]:

# ═════════════════════════════════════════════════════════════
# 5.  PER-IMAGE SEGMENTATION
# ═════════════════════════════════════════════════════════════

def segment_image(image_np: np.ndarray,
                  processor: Sam3Processor,
                  score_threshold: float = SCORE_THRESHOLD,
                  iou_threshold: float   = IOU_THRESHOLD
                  ) -> dict[str, list[dict]]:
    """
    Run SAM3 on one image.

    Returns
    -------
    dict  {super_category → [instance, …]}
    Each instance dict contains: mask, box, score, prompt, sub_cat
    """
    image_pil       = Image.fromarray(image_np)
    inference_state = processor.set_image(image_pil)   # encode once

    # Collect raw detections bucketed by super-category
    buckets: dict[str, list[dict]] = {
        sc: [] for sc in set(SUPER_CATEGORY_MAP.values())
    }

    for sub_cat, prompts in CATEGORY_PROMPTS.items():
        super_cat = SUPER_CATEGORY_MAP[sub_cat]
        for prompt in prompts:
            hits = _query_sam3(processor, inference_state,
                               prompt, score_threshold)
            for h in hits:
                h["sub_cat"] = sub_cat
            buckets[super_cat].extend(hits)

    # De-duplicate within each super-category
    return {
        sc: _deduplicate(insts, iou_threshold)
        for sc, insts in buckets.items()
    }



In [9]:

# ═════════════════════════════════════════════════════════════
# 6.  UNIQUE ID GENERATION
# ═════════════════════════════════════════════════════════════

def _make_uid() -> str:
    """
    Generate a compact, globally unique instance identifier.

    Returns the first 8 hex characters of a UUID4, e.g. 'a3f2c1b0'.
    Collision probability over 10 million instances: < 1 in 10 000.
    If you need absolute zero-collision guarantees, return the full
    uuid.uuid4().hex (32 chars) instead.
    """
    return uuid.uuid4().hex[:8]



In [10]:

# ═════════════════════════════════════════════════════════════
# 7.  MASK SAVING
# ═════════════════════════════════════════════════════════════

def _save_mask(mask_np: np.ndarray,
               frame_stem: str,
               super_cat: str,
               sub_cat: str,
               local_idx: int,
               uid: str,
               score: float,
               image_dir: Path) -> Path:
    """
    Save one binary mask PNG into image_dir.

    File name:
        <frame_stem>_<super_cat>_<sub_cat>_<local_idx:04d>_<uid>_<score:.3f>_mask.png

    The mask is saved as a single-channel (grayscale) PNG:
        255 = object pixel,  0 = background pixel.

    Reload with:
        mask = np.array(Image.open(path)) > 0
    """
    filename = (
        f"{frame_stem}"
        f"_{super_cat}"
        f"_{sub_cat}"
        f"_{local_idx:04d}"
        f"_{uid}"
        f"_{score:.3f}"
        f"_mask.png"
    )
    mask_uint8 = mask_np.astype(np.uint8) * 255
    mask_img   = Image.fromarray(mask_uint8)   # uint8 2-D array → "L" auto-detected
    dest       = image_dir / filename
    mask_img.save(dest)
    return dest


def save_all_masks(results: dict[str, list[dict]],
                   frame_stem: str,
                   image_dir: Path) -> list[Path]:
    """
    Save every instance mask for one image into image_dir.

    For each instance:
      • Generates a UUID4-based unique ID (uid).
      • Assigns a local_idx (monotonically increasing across all categories
        within this image so every mask file inside the folder is unique).
      • Attaches uid, local_idx, and mask_path back onto the instance dict
        (so save_manifest() can read them).

    Returns list of saved mask paths.
    """
    image_dir.mkdir(parents=True, exist_ok=True)
    saved: list[Path] = []
    local_idx = 0

    for super_cat, instances in results.items():
        for inst in instances:
            uid  = _make_uid()
            path = _save_mask(
                mask_np    = inst["mask"],
                frame_stem = frame_stem,
                super_cat  = super_cat,
                sub_cat    = inst["sub_cat"],
                local_idx  = local_idx,
                uid        = uid,
                score      = inst["score"],
                image_dir  = image_dir,
            )
            # Annotate the instance dict for downstream use (vis, manifest)
            inst["uid"]        = uid
            inst["local_idx"]  = local_idx
            inst["mask_path"]  = path
            saved.append(path)
            local_idx += 1

    return saved



In [11]:

# ═════════════════════════════════════════════════════════════
# 8.  PER-FRAME MANIFEST (JSON index)
# ═════════════════════════════════════════════════════════════

def save_manifest(results: dict[str, list[dict]],
                  frame_stem: str,
                  image_dir: Path) -> Path:
    """
    Write a JSON index of all instances detected in this frame.

    The manifest is saved as:
        image_dir / <frame_stem>_manifest.json

    Format
    ------
    [
      {
        "uid":       "a3f2c1b0",
        "local_idx": 0,
        "super_cat": "vehicle",
        "sub_cat":   "vehicle_headlights",
        "score":     0.923,
        "box":       [x1, y1, x2, y2],
        "mask_file": "<frame>_vehicle_vehicle_headlights_0000_a3f2c1b0_0.923_mask.png"
      },
      ...
    ]

    Notes
    -----
    • Call this AFTER save_all_masks() so that inst["uid"], inst["local_idx"],
      and inst["mask_path"] have already been populated.
    • Bounding box values are rounded to 2 decimal places.
    """
    records = []
    for super_cat, instances in results.items():
        for inst in instances:
            records.append({
                "uid":       inst["uid"],
                "local_idx": inst["local_idx"],
                "super_cat": super_cat,
                "sub_cat":   inst["sub_cat"],
                "score":     round(inst["score"], 4),
                "box":       [round(float(v), 2) for v in inst["box"]],
                "mask_file": inst["mask_path"].name,
            })

    # Sort by local_idx for readability
    records.sort(key=lambda r: r["local_idx"])

    manifest_path = image_dir / f"{frame_stem}_manifest.json"
    with open(manifest_path, "w") as fh:
        json.dump(records, fh, indent=2)

    return manifest_path



In [12]:

# ═════════════════════════════════════════════════════════════
# 9.  VISUALISATION  (optional, saved as overview.jpg)
# ═════════════════════════════════════════════════════════════

def save_visualisation(image_np: np.ndarray,
                        results: dict[str, list[dict]],
                        frame_stem: str,
                        image_dir: Path,
                        alpha: float = 0.40) -> None:
    """
    Overlay all masks on the original image and save as overview.jpg.
    Bounding boxes and labels (sub_cat / score / uid) are drawn on top.

    Note: Call after save_all_masks() so inst["uid"] is available.
    """
    overlay = image_np.copy().astype(np.float32)

    for super_cat, instances in results.items():
        color = VIS_COLORS.get(super_cat, (200, 200, 200))
        for inst in instances:
            for c, v in enumerate(color):
                overlay[:, :, c][inst["mask"]] = (
                    overlay[:, :, c][inst["mask"]] * (1 - alpha) + v * alpha
                )

    fig, ax = plt.subplots(figsize=(14, 6), dpi=100)
    ax.imshow(overlay.astype(np.uint8))
    ax.axis("off")

    for super_cat, instances in results.items():
        cnorm = tuple(v / 255 for v in VIS_COLORS.get(super_cat, (200, 200, 200)))
        for inst in instances:
            x1, y1, x2, y2 = inst["box"]
            ax.add_patch(mpatches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=1.2, edgecolor=cnorm, facecolor="none"
            ))
            label = (
                f"{inst['sub_cat']}  "
                f"{inst['score']:.2f}  "
                f"#{inst['local_idx']:04d}"
                f"  [{inst.get('uid', '?')}]"
            )
            ax.text(x1, max(y1 - 4, 0), label,
                    color="white", fontsize=6,
                    bbox=dict(facecolor=cnorm, alpha=0.75, pad=1))

    legend = [
        mpatches.Patch(color=tuple(v / 255 for v in c), label=cat)
        for cat, c in VIS_COLORS.items()
    ]
    ax.legend(handles=legend, loc="upper right", fontsize=8)
    ax.set_title(f"SAM3 segmentation — {frame_stem}", fontsize=10)
    plt.tight_layout(pad=0)

    # Render to an in-memory PNG buffer, then save as JPEG via Pillow.
    # This avoids FigureCanvasAgg.print_jpg() which dropped the `quality`
    # kwarg in newer matplotlib builds.
    import io as _io
    buf = _io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    out_path = image_dir / f"{frame_stem}_overview.jpg"
    Image.open(buf).convert("RGB").save(str(out_path), format="JPEG", quality=92)



In [13]:

# ═════════════════════════════════════════════════════════════
# 10. KITTI EIGEN-SPLIT DIRECTORY CRAWLER
# ═════════════════════════════════════════════════════════════

def iter_kitti_images(input_root: Path) -> list[tuple[Path, Path]]:
    """
    Walk the KITTI eigen-split dataset and collect all RGB image files.

    Expected layout
    ---------------
        input_root/
        └── <seq_date_drive>/         level-1  (e.g. 2011_09_26_drive_0001_sync)
            └── <date>/               level-2  (e.g. 2011_09_26)
                └── <seq>/            level-3  (e.g. 0000000001)
                    └── image_02/
                        └── data/
                            ├── 0000000000.png
                            └── ...

    This matches the glob pattern:
        input_root/*/*/*/image_02/data/*.png

    The matching LiDAR depth maps live at:
        input_root/*/*/*/proj_depth/groundtruth/image_02/*.png
    They share the same frame stem as the RGB files and can be
    correlated via the manifest.

    Returns
    -------
    Sorted list of (rel_data_dir, img_path) tuples where:
      • rel_data_dir  – Path RELATIVE to input_root of the containing
                        `image_02/data` directory.  This is used to
                        mirror the exact same subdirectory tree under
                        output_root.
                        Example: Path("2011_09_26_drive_0001_sync/2011_09_26/
                                       0000000001/image_02/data")
      • img_path      – absolute Path to the PNG file.
    """
    pairs: list[tuple[Path, Path]] = []

    for img_path in sorted(input_root.glob(EIGEN_GLOB_PATTERN)):
        if img_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        # rel_data_dir = e.g. 2011_09_26_drive_0001_sync/2011_09_26/
        #                      0000000001/image_02/data
        rel_data_dir = img_path.parent.relative_to(input_root)
        pairs.append((rel_data_dir, img_path))

    return pairs


def _depth_path_for(input_root: Path, rel_data_dir: Path,
                    frame_stem: str) -> Path | None:
    """
    Return the LiDAR ground-truth depth PNG that matches a given RGB frame,
    or None if it does not exist.

    Mapping:
        <l1>/<l2>/<l3>/image_02/data/<stem>.png   (RGB)
        <l1>/<l2>/<l3>/proj_depth/groundtruth/image_02/<stem>.png  (depth)

    rel_data_dir = <l1>/<l2>/<l3>/image_02/data
    """
    # Navigate up to the sequence root (<l1>/<l2>/<l3>)
    seq_root = input_root / rel_data_dir.parents[2]   # 3 levels up from "data"
    depth_path = (
        seq_root / "proj_depth" / "groundtruth" / "image_02"
        / f"{frame_stem}.png"
    )
    return depth_path if depth_path.exists() else None



In [14]:

# ═════════════════════════════════════════════════════════════
# 11. BATCH RUNNER
# ═════════════════════════════════════════════════════════════

def _is_frame_complete(frame_out_dir: Path, frame_stem: str) -> bool:
    """
    Return True only when a frame was fully processed.

    The manifest JSON is the very last artefact written by the pipeline
    (after all mask PNGs have been saved), so its presence guarantees
    that processing finished cleanly.  A directory that exists but has
    no manifest indicates an interrupted run — treat it as incomplete.
    """
    return (frame_out_dir / f"{frame_stem}_manifest.json").exists()


def run_batch(input_root: str | Path,
              output_root: str | Path,
              score_threshold: float = SCORE_THRESHOLD,
              iou_threshold:   float = IOU_THRESHOLD,
              save_vis:        bool  = True) -> None:
    """
    Process the entire KITTI eigen-split dataset and write instance masks.

    Output layout (mirrors input exactly)
    --------------------------------------
    output_root/
    └── <seq_date_drive>/<date>/<seq>/image_02/data/
        └── <frame_stem>/
            ├── <frame_stem>_<super>_<sub>_<idx:04d>_<uid8>_<score>_mask.png
            ├── ...
            ├── <frame_stem>_manifest.json      (always written)
            └── <frame_stem>_overview.jpg       (if save_vis=True)

    Parameters
    ----------
    input_root      : Root of the KITTI eigen-split dataset.
    output_root     : Where to write all output masks.
    score_threshold : SAM3 confidence cutoff (0–1).
    iou_threshold   : NMS overlap threshold for de-duplication (0–1).
    save_vis        : Whether to save a coloured overview JPEG per frame.

    Resume behaviour
    ----------------
    A frame is considered complete if its manifest JSON is present.
    • Complete  → silently skipped (counted in the progress bar start).
    • Partial   → leftover directory is deleted, then re-processed from
                  scratch so no corrupt/incomplete masks are kept.
    """
    input_root  = Path(input_root)
    output_root = Path(output_root)

    if not input_root.exists():
        raise FileNotFoundError(f"Input root not found: {input_root}")

    output_root.mkdir(parents=True, exist_ok=True)

    # ── Discover images ───────────────────────────────────────
    image_pairs = iter_kitti_images(input_root)
    if not image_pairs:
        log.warning(
            "No images found under %s using pattern '%s'.",
            input_root, EIGEN_GLOB_PATTERN,
        )
        return

    n_total      = len(image_pairs)
    unique_seqs  = len({rel for rel, _ in image_pairs})
    log.info("Found %d image(s) across %d sequence(s).",
             n_total, unique_seqs)

    # ── Pre-scan: count already-completed frames ──────────────
    # This lets us (a) fast-forward the progress bar, and (b) log a
    # clear summary before the heavy work begins.
    n_already_done = sum(
        1 for rel, img in image_pairs
        if _is_frame_complete(output_root / rel / img.stem, img.stem)
    )
    n_to_process = n_total - n_already_done

    log.info("Already completed : %d / %d", n_already_done, n_total)
    log.info("Remaining         : %d",        n_to_process)

    if n_to_process == 0:
        log.info("Nothing to do — all frames already processed.")
        return

    # ── Load SAM3 (once) ──────────────────────────────────────
    _, processor = load_sam3(CACHE_DIR)

    # ── Statistics ────────────────────────────────────────────
    total_masks = 0
    n_processed = 0     # newly processed this run
    n_skipped   = n_already_done
    errors      = 0

    # ── Progress bar ──────────────────────────────────────────
    # • total   = all frames (so ETA and % reflect overall dataset progress)
    # • initial = already-completed count (bar starts where we left off)
    # • set_postfix shows live counters: new / skipped / err / masks
    pbar = tqdm(
        total        = n_total,
        initial      = n_already_done,
        desc         = "SAM3 · KITTI eigen",
        unit         = " img",
        dynamic_ncols= True,
        colour       = "green",
        smoothing    = 0.05,    # smoother ETA for long batches
    )
    pbar.set_postfix(
        new     = n_processed,
        skipped = n_skipped,
        err     = errors,
        masks   = total_masks,
        refresh = False,
    )

    # ── Main loop ─────────────────────────────────────────────
    for img_idx, (rel_data_dir, img_path) in enumerate(image_pairs, start=1):

        frame_stem    = img_path.stem
        frame_out_dir = output_root / rel_data_dir / frame_stem

        # ── Resume / skip logic ───────────────────────────────
        if _is_frame_complete(frame_out_dir, frame_stem):
            # Already fully processed in a previous run — do not update
            # the bar here; it was pre-counted in  above.
            continue

        if frame_out_dir.exists():
            # Partial output from an interrupted run: delete and redo.
            log.warning(
                "[%d/%d]  Partial output detected for '%s' — removing and reprocessing.",
                img_idx, n_total, frame_stem,
            )
            shutil.rmtree(frame_out_dir)
        else:
            log.info("[%d/%d]  %s / %s",
                     img_idx, n_total, rel_data_dir, img_path.name)

        # ── Optional: log matched depth path ─────────────────
        depth_p = _depth_path_for(input_root, rel_data_dir, frame_stem)
        if depth_p:
            log.debug("  ├─ depth : %s", depth_p)
        else:
            log.debug("  ├─ depth : (not found)")

        try:
            # Load RGB image
            image_np = np.array(Image.open(img_path).convert("RGB"))

            # Run SAM3
            results = segment_image(
                image_np        = image_np,
                processor       = processor,
                score_threshold = score_threshold,
                iou_threshold   = iou_threshold,
            )

            # Save binary mask PNGs + attach uid/local_idx/mask_path to insts
            saved = save_all_masks(
                results    = results,
                frame_stem = frame_stem,
                image_dir  = frame_out_dir,
            )
            total_masks += len(saved)

            # Save per-frame manifest JSON  ← written LAST (completion marker)
            manifest_p = save_manifest(
                results    = results,
                frame_stem = frame_stem,
                image_dir  = frame_out_dir,
            )

            n_processed += 1

            # Per-image summary log
            for super_cat, instances in results.items():
                if instances:
                    log.info("  ├─ %-22s  %d instance(s)",
                             super_cat, len(instances))
            log.info("  ├─ %d mask(s) saved", len(saved))
            log.info("  ├─ manifest  → %s", manifest_p.name)
            log.info("  └─ output dir → %s", frame_out_dir)

        except Exception as exc:
            log.error("  ERROR processing %s: %s", img_path, exc,
                      exc_info=True)
            errors += 1
            # Advance the bar even on error so total stays consistent
            pbar.update(1)
            pbar.set_postfix(
                new     = n_processed,
                skipped = n_skipped,
                err     = errors,
                masks   = total_masks,
                refresh = False,
            )
            continue

        # Optional visualisation — isolated so a vis failure never discards
        # already-saved masks or inflates the error counter.
        if save_vis:
            try:
                save_visualisation(
                    image_np   = image_np,
                    results    = results,
                    frame_stem = frame_stem,
                    image_dir  = frame_out_dir,
                )
            except Exception as vis_exc:
                log.warning("  Visualisation failed for %s (masks are safe): %s",
                            img_path.name, vis_exc)

        # Advance progress bar and refresh live stats
        pbar.update(1)
        pbar.set_postfix(
            new     = n_processed,
            skipped = n_skipped,
            err     = errors,
            masks   = total_masks,
            refresh = False,
        )

    pbar.close()

    # ── Final report ──────────────────────────────────────────
    log.info("═" * 60)
    log.info("Batch complete.")
    log.info("  Frames in dataset   : %d", n_total)
    log.info("  Pre-existing (skip) : %d", n_skipped)
    log.info("  Newly processed     : %d", n_processed)
    log.info("  Errors              : %d", errors)
    log.info("  Total masks saved   : %d", total_masks)
    log.info("  Output root         : %s", output_root)



In [15]:

# ═════════════════════════════════════════════════════════════
# 12. LOADING UTILITY  (for downstream consumers)
# ═════════════════════════════════════════════════════════════

def load_masks_for_frame(output_root: str | Path,
                          rel_data_dir: str | Path,
                          frame_stem: str,
                          super_cat_filter: str | None = None,
                          sub_cat_filter:   str | None = None,
                          use_manifest:     bool       = True,
                          ) -> list[dict]:
    """
    Reconstruct the instance list for one frame from saved mask PNGs.

    Prefers the JSON manifest (faster, no filename parsing) but falls back
    to glob+filename parsing when the manifest is absent.

    Parameters
    ----------
    output_root      : Root of the saved masks (same as passed to run_batch).
    rel_data_dir     : Relative path to the image_02/data directory,
                       e.g. "2011_09_26_drive_0001_sync/2011_09_26/
                             0000000001/image_02/data"
    frame_stem       : Image filename stem, e.g. "0000000005".
    super_cat_filter : If given, return only instances of this super-category.
    sub_cat_filter   : If given, return only instances of this sub-category.
    use_manifest     : If True (default), read from the JSON manifest.
                       Set False to fall back to glob+filename parsing.

    Returns
    -------
    List of dicts, each with keys:
        uid, local_idx, super_cat, sub_cat, score, box, mask (H×W bool), path
    """
    frame_dir = Path(output_root) / rel_data_dir / frame_stem
    if not frame_dir.exists():
        raise FileNotFoundError(f"Frame mask directory not found: {frame_dir}")

    # ── Fast path: use manifest ───────────────────────────────
    manifest_path = frame_dir / f"{frame_stem}_manifest.json"
    if use_manifest and manifest_path.exists():
        with open(manifest_path) as fh:
            records = json.load(fh)

        instances = []
        for rec in records:
            if super_cat_filter and rec["super_cat"] != super_cat_filter:
                continue
            if sub_cat_filter and rec["sub_cat"] != sub_cat_filter:
                continue
            mask_file = frame_dir / rec["mask_file"]
            if not mask_file.exists():
                log.warning("Mask file missing: %s", mask_file)
                continue
            mask_np = np.array(Image.open(mask_file).convert("L")) > 127
            instances.append({
                "uid":       rec["uid"],
                "local_idx": rec["local_idx"],
                "super_cat": rec["super_cat"],
                "sub_cat":   rec["sub_cat"],
                "score":     rec["score"],
                "box":       rec["box"],
                "mask":      mask_np,
                "path":      mask_file,
            })
        return instances

    # ── Fallback: glob + filename parsing ─────────────────────
    # Expected filename format:
    #   <frame_stem>_<super_cat>_<sub_cat>_<local_idx:04d>_<uid8>_<score:.3f>_mask.png
    instances = []
    for mask_file in sorted(frame_dir.glob("*_mask.png")):
        stem = mask_file.stem                              # strip .png
        stem = stem[:-5]                                   # strip "_mask"
        if not stem.startswith(frame_stem + "_"):
            continue
        suffix = stem[len(frame_stem) + 1:]               # strip "<frame_stem>_"

        # Split from the right: [categories_part, local_idx, uid, score]
        parts = suffix.rsplit("_", 3)
        if len(parts) != 4:
            log.warning("Unexpected filename format: %s", mask_file.name)
            continue

        cat_str, idx_str, uid_str, score_str = parts

        # cat_str = "<super_cat>_<sub_cat>", e.g. "vehicle_vehicle_headlights"
        # Split on first underscore to separate super_cat from sub_cat
        cat_parts = cat_str.split("_", 1)
        if len(cat_parts) != 2:
            log.warning("Cannot parse categories from: %s", cat_str)
            continue
        super_cat, sub_cat = cat_parts

        if super_cat_filter and super_cat != super_cat_filter:
            continue
        if sub_cat_filter and sub_cat != sub_cat_filter:
            continue

        try:
            local_idx = int(idx_str)
            score     = float(score_str)
        except ValueError:
            log.warning("Cannot parse idx/score from: %s", mask_file.name)
            continue

        mask_np = np.array(Image.open(mask_file).convert("L")) > 127
        instances.append({
            "uid":       uid_str,
            "local_idx": local_idx,
            "super_cat": super_cat,
            "sub_cat":   sub_cat,
            "score":     score,
            "box":       None,        # not available without manifest
            "mask":      mask_np,
            "path":      mask_file,
        })

    instances.sort(key=lambda x: x["local_idx"])
    return instances



In [16]:

# ═════════════════════════════════════════════════════════════
# 13. ENTRY POINT
# ═════════════════════════════════════════════════════════════

#if __name__ == "__main__":
"""
    Usage:
        python inst_seg_kitti_eigen.py <input_root> <output_root> [OPTIONS]

    Examples:
        # Default thresholds, with visualisation
        python inst_seg_kitti_eigen.py \\
            /STORAGE/Arka/Downloads/KITTI/kitti_eigen_split_dataset \\
            /STORAGE/Arka/Downloads/KITTI/kitti_eigen_masks

        # Custom thresholds, no visualisation
        python inst_seg_kitti_eigen.py \\
            /STORAGE/Arka/Downloads/KITTI/kitti_eigen_split_dataset \\
            /STORAGE/Arka/Downloads/KITTI/kitti_eigen_masks \\
            --score 0.55 --iou 0.65 --no-vis

    Loading masks later:
        from inst_seg_kitti_eigen import load_masks_for_frame
        instances = load_masks_for_frame(
            output_root  = "/STORAGE/Arka/Downloads/KITTI/kitti_eigen_masks",
            rel_data_dir = "2011_09_26_drive_0001_sync/2011_09_26/"
                           "0000000001/image_02/data",
            frame_stem   = "0000000005",
            super_cat_filter = "vehicle",   # optional
        )
        for inst in instances:
            mask  = inst["mask"]        # H×W bool ndarray
            uid   = inst["uid"]         # "a3f2c1b0"  — globally unique ID
            score = inst["score"]       # float
            sub   = inst["sub_cat"]     # e.g. "vehicle_headlights"
"""
"""
    import argparse

    parser = argparse.ArgumentParser(
        description="SAM3 batch instance segmentation for KITTI eigen-split dataset"
    )
    parser.add_argument(
        "input_root",
        help="Root of the KITTI eigen-split dataset "
             "(contains <seq_date_drive>/<date>/<seq>/image_02/data/)",
    )
    parser.add_argument(
        "output_root",
        help="Root output folder; directory tree mirrors input_root",
    )
    parser.add_argument(
        "--score", type=float, default=SCORE_THRESHOLD,
        help=f"SAM3 confidence threshold (default: {SCORE_THRESHOLD})",
    )
    parser.add_argument(
        "--iou", type=float, default=IOU_THRESHOLD,
        help=f"NMS IoU threshold (default: {IOU_THRESHOLD})",
    )
    parser.add_argument(
        "--no-vis", action="store_true",
        help="Skip saving overview visualisation JPEGs",
    )
    args = parser.parse_args()
"""

'\n    import argparse\n\n    parser = argparse.ArgumentParser(\n        description="SAM3 batch instance segmentation for KITTI eigen-split dataset"\n    )\n    parser.add_argument(\n        "input_root",\n        help="Root of the KITTI eigen-split dataset "\n             "(contains <seq_date_drive>/<date>/<seq>/image_02/data/)",\n    )\n    parser.add_argument(\n        "output_root",\n        help="Root output folder; directory tree mirrors input_root",\n    )\n    parser.add_argument(\n        "--score", type=float, default=SCORE_THRESHOLD,\n        help=f"SAM3 confidence threshold (default: {SCORE_THRESHOLD})",\n    )\n    parser.add_argument(\n        "--iou", type=float, default=IOU_THRESHOLD,\n        help=f"NMS IoU threshold (default: {IOU_THRESHOLD})",\n    )\n    parser.add_argument(\n        "--no-vis", action="store_true",\n        help="Skip saving overview visualisation JPEGs",\n    )\n    args = parser.parse_args()\n'

In [17]:
run_batch(
input_root      = r"/STORAGE/Arka/Downloads/KITTI/kitti_eigen_split_dataset_train_2/",
output_root     = r"/STORAGE/Arka/a_MDE_Code/1_Anaysis/Output/",
score_threshold = SCORE_THRESHOLD,
iou_threshold   = IOU_THRESHOLD,
save_vis        = True,
)

SAM3 · KITTI eigen:  31%|###1      | 2846/9059 [00:00<?, ? img/s]

KeyboardInterrupt: 